# Historical NBA Data Pipeline

This notebook builds the historical NBA dataset used for predictive modeling.

It collects game-level statistics, cleans and standardizes the data, generates team-level rolling and derived features, and prepares datasets for model training and future game predictions.

## Pipeline Overview

1. Historical game data collection
2. Data cleaning and standardization
3. Team-level rolling feature engineering
4. Derived feature engineering
5. Historical training dataset preparation
6. Future prediction dataset preparation

## 1. Historical Game Data Collection

Collects historical NBA game-level statistics from Basketball Reference and stores the raw data for further processing.

In [ ]:
#Создаем raw_games.csv

import os
import time
import random
import logging
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup, Comment
import cloudscraper

# ---------------- CONFIG ---------------- #

BASE = "https://www.basketball-reference.com"
SEASONS = ["2026"
           #,"2025","2024","2023","2022","2021","2020","2019","2018","2017","2016","2015","2014","2013","2012","2011"
          ]
TEAMS = [
    "ATL","BOS","BRK","CHI","CHO","CLE","DAL","DEN","DET",
    "GSW","HOU","IND","LAC","LAL","MEM","MIA","MIL","MIN",
    "NOP","NYK","OKC","ORL","PHI","PHO","POR","SAC","SAS",
    "TOR","UTA","WAS"
]

OUTPUT_FILE = "raw_games.csv"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

# ---------------- SCRAPER ---------------- #

session = cloudscraper.create_scraper(
    browser={
        "browser": "chrome",
        "platform": "windows",
        "desktop": True,
    }
)

def safe_sleep():
    time.sleep(5 + random.random() * 4)

def get_soup(url):
    r = session.get(url, timeout=30)

    if r.status_code in (403, 429):
        logging.error(f"{r.status_code} BLOCK → {url}")
        time.sleep(60)
        return None

    if r.status_code != 200:
        logging.error(f"{r.status_code} → {url}")
        return None

    return BeautifulSoup(r.text, "lxml")

# ---------------- TABLE PARSING ---------------- #

RENAME_MAP = {
    "date": "game_date",
    "opp_name_abbr": "Opp",
    "team_game_num": "G"
}

def extract_table(soup, table_id):
    if soup is None:
        return None

    table = soup.find("table", id=table_id)
    if not table:
        for c in soup.find_all(string=lambda t: isinstance(t, Comment)):
            if table_id in c:
                cs = BeautifulSoup(c, "lxml")
                table = cs.find("table", id=table_id)
                break

    if not table:
        return None

    rows = table.find("tbody").find_all("tr")
    data = []

    for row in rows:
        if row.get("class") and "thead" in row.get("class"):
            continue

        rec = {}
        for cell in row.find_all(["th", "td"]):
            stat = cell.get("data-stat")
            if stat:
                rec[stat] = cell.text.strip()

        if rec:
            data.append(rec)

    if not data:
        return None

    df = pd.DataFrame(data)
    df.rename(columns=RENAME_MAP, inplace=True)
    return df

def make_event_key(date, team, opp, is_home):
    if (
        pd.isna(date)
        or pd.isna(team)
        or pd.isna(opp)
        or pd.isna(is_home)
    ):
        return np.nan

    try:
        d = pd.to_datetime(date, utc=True).strftime("%Y-%m-%d")
    except Exception:
        return np.nan

    if is_home:
        home, away = str(team), str(opp)
    else:
        home, away = str(opp), str(team)

    return f"{d}_{home}_{away}"



# ---------------- GAMELOGS ---------------- #

def fetch_team_season(team_id, season):

    base_url = f"{BASE}/teams/{team_id}/{season}/gamelog/"
    adv_url  = f"{BASE}/teams/{team_id}/{season}/gamelog-advanced/"

    soup_base = get_soup(base_url)
    safe_sleep()
    soup_adv  = get_soup(adv_url)

    base_parts = []
    adv_parts  = []

    for tid in ["team_game_log_reg", "team_game_log_post"]:
        df = extract_table(soup_base, tid)
        if df is not None:
            base_parts.append(df)

    for tid in ["team_game_log_adv_reg", "team_game_log_adv_post"]:
        df = extract_table(soup_adv, tid)
        if df is not None:
            adv_parts.append(df)

    if not base_parts:
        logging.info(f"⏭ no BASIC for {team_id} {season}")
        return None

    if not adv_parts:
        logging.info(f"⏭ no ADV for {team_id} {season}")
        return None

    base_df = pd.concat(base_parts, ignore_index=True)
    adv_df  = pd.concat(adv_parts, ignore_index=True)

    # 🔑 SAFE MERGE
    keys = ["game_date", "Opp"]
    for k in keys:
        if k not in base_df.columns or k not in adv_df.columns:
            logging.error(f"{team_id} {season} missing key {k}")
            return None

    df = pd.merge(
        base_df,
        adv_df,
        on=keys,
        how="inner",
        suffixes=("_base", "_adv")
    )
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

    df["team_id"] = team_id
    df["season"] = season
    df = df.rename(columns={"game_location_base": "Is_Home"})

    df["event_key"] = df.apply(
        lambda r: make_event_key(
            r["game_date"],
            r["team_id"],
            r["Opp"],
            r["Is_Home"]
        ),
        axis=1
    )

    logging.info(f"✅ {team_id} {season} games={len(df)}")
    return df

# ---------------- MAIN ---------------- #

def run_incremental():
    # Считаем существующий файл, если есть
    if os.path.exists(OUTPUT_FILE):
        existing_df = pd.read_csv(OUTPUT_FILE)
        EXPECTED_COLS = list(existing_df.columns)

 #       assert list(existing_df.columns) == list(new_dataset.columns), \
 #           "SCHEMA MISMATCH: columns order or count differs"
        
        existing_df["game_date"] = pd.to_datetime(existing_df["game_date"], errors="coerce")
        
        # 🔁 MIGRATION: если event_key ещё нет — создаём
        if "event_key" not in existing_df.columns:
            print("⚠️ event_key not found in existing file — migrating schema")
        
            existing_df["event_key"] = existing_df.apply(
                lambda r: make_event_key(
                    r["game_date"],
                    r["team_id"],
                    r["Opp"]                    
                ),
                axis=1
            )

        # сохраняем миграцию
        existing_df.to_csv(OUTPUT_FILE, index=False)
        
        existing_df["event_key"] = existing_df["event_key"].astype(str)
        existing_keys = set(existing_df["event_key"])

        logging.info(f"Loaded existing data: {len(existing_df)} rows")
    else:
        existing_df = pd.DataFrame()
        existing_keys = set()

    all_new_games = []

    for team_id in TEAMS:
        for season in SEASONS:
            try:
                df = fetch_team_season(team_id, season)
                if df is None or df.empty:
                    continue

                # фильтруем только новые матчи
                df["game_date"] = pd.to_datetime(df["game_date"])
                df["key"] = list(zip(df["team_id"], df["game_date"]))
                df_new = df[~df["event_key"].isin(existing_keys)].copy()
                df_new.drop(columns="key", inplace=True)

                if not df_new.empty:
                    all_new_games.append(df_new)
                    logging.info(f"{team_id} {season} new rows={len(df_new)}")

            except Exception as e:
                logging.error(f"{team_id} {season} FAIL: {e}")

    if not all_new_games:
        logging.info("No new games to append.")
        return

    new_dataset = pd.concat(all_new_games, ignore_index=True)
    new_dataset = new_dataset.sort_values("game_date")
 
    # Если файла нет, пишем с header
    if not os.path.exists(OUTPUT_FILE):
        new_dataset.to_csv(OUTPUT_FILE, index=False)
    else:
        new_dataset = new_dataset[existing_df.columns]
        new_dataset.to_csv(OUTPUT_FILE, index=False, mode="a", header=False)

    logging.info(f"🎉 DONE, appended rows={len(new_dataset)}. Total rows now ~{len(existing_df)+len(new_dataset)}")

# ---------------- RUN ---------------- #

if __name__ == "__main__":
    run_incremental()


## 2. Data Cleaning and Standardization

Cleans raw game records, standardizes team names and dates, and prepares consistent game-level data.

In [ ]:
#создаем cleaned_games.csv

import time
import random
import logging
import pandas as pd
from bs4 import BeautifulSoup, Comment
import cloudscraper


data = "raw_games.csv"

RENAME_MAP = {
    "game_date": "Date",
    "Opp": "Opp_name",
    "team_game_score_base": "PTS_for",
    "opp_team_game_score_base": "PTS_against",
    "fg": "FG",
    "fga": "FGA",
    "fg_pct": "FG%",
    "fg3": "3P",
    "fg3a": "3PA",
    "fg3_pct": "3P%",
    "fg2": "2P",
    "fg2a": "2PA",
    "fg2_pct": "2P%",
    "efg_pct_base": "eFG%",
    "ft": "FT",
    "fta": "FTA",
    "ft_pct": "FT%",
    "orb": "ORB",
    "drb": "DRB",
    "trb": "TRB",
    "ast": "AST",
    "stl": "STL",
    "blk": "BLK",
    "tov": "TOV",
    "pf": "PF",
    "opp_fg": "FG_against",
    "opp_fga": "FGA_against",
    "opp_fg_pct": "FG%_against",
    "opp_fg3": "3P_against",
    "opp_fg3a": "3PA_against",
    "opp_fg3_pct": "3P%_against",
    "opp_fg2": "2P_against",
    "opp_fg2a": "2PA_against",
    "opp_fg2_pct": "2P%_against",
    "opp_ft": "FT_against",
    "opp_fta": "FTA_against",
    "opp_ft_pct": "FT%_against",
    "opp_orb": "ORB_against",
    "opp_drb": "DRB_against",
    "opp_trb": "TRB_against",
    "opp_ast": "AST_against",
    "opp_stl": "STL_against",
    "opp_blk": "BLK_against",
    "opp_tov": "TOV_against",
    "opp_pf": "PF_against",
    "team_off_rtg": "ORtg",
    "team_def_rtg":	"DRtg",
    "pace": "Pace",
    "fta_per_fga_pct": "FTr",
    "ts_pct": "TS%",	
    "team_trb_pct"	: "TRB%",
    "team_ast_pct"	: "AST%",
    "team_stl_pct"	: "STL%",
    "team_blk_pct": "BLK%",
    "team_tov_pct"	: "TOV%",
    "team_orb_pct"	: "ORB%",
    "ft_rate"	: "FT/FGA",
    "opp_efg_pct_adv"	: "eFG%_against",
    "opp_tov_pct"	: "TOV%_against",
    "opp_orb_pct"	: "ORB%_against",
    "opp_ft_rate":	"FT/FGA_against",
    "team_id": "Team"
}


df = pd.read_csv(data)
df.rename(columns=RENAME_MAP, inplace=True)

# === CANONICAL TYPES ===
df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
df["Team"] = df["Team"].astype(str)
df["Opp_name"] = df["Opp_name"].astype(str)

df = df.sort_values(["season", "Team", "Date"])


TEAM_MAP_REVERSE = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "BRK": "Brooklyn Nets",
    "CHO": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "LA Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHO": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
}

df["Is_Home"] = ~df["Is_Home"].str.contains("@", na=False)
df["Team"] = df["Team"].map(TEAM_MAP_REVERSE)
df["Opp_name"] = df["Opp_name"].map(TEAM_MAP_REVERSE)

df.drop(columns=["event_key"], inplace=True, errors="ignore")

df["event_key"] = df.apply(
    lambda r: make_event_key(
        r["Date"],
        r["Team"],
        r["Opp_name"],
        r["Is_Home"]
    ),
    axis=1
)


df["Prev_Date"] = df.groupby(["season", "Team"])["Date"].shift(1)
df["HoursRest"] = (df["Date"] - df["Prev_Date"]).dt.total_seconds() / 3600
df["BackToBack"] = (df["HoursRest"] <= 24).astype(int)
df["ShortRest"] = ((df["HoursRest"] > 24) & (df["HoursRest"] <= 48)).astype(int)
df["LongRest"] = (df["HoursRest"] >= 72).astype(int)


df["PTS_for"] = pd.to_numeric(df["PTS_for"], errors="coerce")
df["PTS_against"] = pd.to_numeric(df["PTS_against"], errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)
df["Total"] = df["PTS_for"] + df["PTS_against"]
df["points_diff"] = df["PTS_for"] - df["PTS_against"]

cols = [
    "event_key",
    "Date",
    "season",    
    "Team",
    "Opp_name",
    "Is_Home",

    "PTS_for",
    "PTS_against",
    "Total",
    "points_diff",
     "FG",
     "FGA",
     "FG%",
     "3P",
     "3PA",
    "3P%",
     "2P",
     "2PA",
     "2P%",
    "eFG%",
     "FT",
     "FTA",
     "FT%",
    "ORB",
    "DRB",
     "TRB",
     "AST",
     "STL",
    "BLK",
    "TOV",
    "PF",
    "ORtg",
    "DRtg",
    "Pace",
    "FT/FGA",
     "TS%",	
     "TRB%",
     "AST%",
     "STL%",
     "BLK%",
     "TOV%",
     "ORB%",
     "FTr",
      "FG_against",
      "FGA_against",
      "FG%_against",
     "3P_against",
     "3PA_against",
     "3P%_against",
     "2P_against",
     "2PA_against",
     "2P%_against",
    "eFG%_against",
     "FT_against",
    "FTA_against",
     "FT%_against",
    "ORB_against",
    "DRB_against",
     "TRB_against",
     "AST_against",
     "STL_against",
     "BLK_against",
    "TOV_against",
     "PF_against",
     "TOV%_against",
     "ORB%_against",
    "FT/FGA_against",

    "Prev_Date",
    "HoursRest",
    "LongRest",
    "ShortRest",
    "BackToBack"
]
for c in cols:
    if c not in df.columns:
        df[c] = pd.NA

df_full = df[cols].copy()

df_full = df_full.dropna(subset="event_key")

df_full.to_csv("cleaned_games.csv", index=False)
df_full.to_parquet("cleaned_games.parquet", compression="zstd")
print("Done")


## 3. Team-Level Rolling Features

Calculates historical team performance indicators using rolling windows. Features are shifted to prevent future information from entering pre-game predictions.

In [ ]:
# создаем team_game_features.csv

import pandas as pd

# ================== LOAD ==================
df = pd.read_csv("cleaned_games.csv", parse_dates=["Date"])

# ================== SORT (CRITICAL) ==================
df = (
    df
    .sort_values(["Team", "season", "Date"])
    .reset_index(drop=True)
)

# =====================================================
# VOLATILITY FEATURES (STD) — PRE-GAME ONLY
# =====================================================

# --- PTS volatility ---
for col, name in [
    ("PTS_for", "Rolling_PTS_for_std"),
    ("PTS_against", "Rolling_PTS_against_std"),
]:
    for window, min_p in [(5, 3), (15, 5)]:
        df[f"{name}_{window}"] = (
            df
            .groupby(["Team", "season"])[col]
            .rolling(window=window, min_periods=min_p)
            .std()
            .shift(1)
            .reset_index(level=[0, 1], drop=True)
        )

# --- Pace volatility ---
df["Rolling_Pace_std_5"] = (
    df
    .groupby(["Team", "season"])["Pace"]
    .rolling(window=5, min_periods=3)
    .std()
    .shift(1)
    .reset_index(level=[0, 1], drop=True)
)


# =====================================================
# CONFIG
# =====================================================
rolling_windows = [5, 15]

stats_columns = [
    "PTS_for",  "FG",
     "FGA",
     "FG%",
     "3P",
     "3PA",
    "3P%",
     "2P",
     "2PA",
     "2P%",
    "eFG%",
     "FT",
     "FTA",
     "FT%",
    "ORB",
    "DRB",
     "TRB",
     "AST",
     "STL",
    "BLK",
    "TOV",
    "PF",
    "ORtg",
    "DRtg",
    "Pace",
    "FT/FGA",
     "TS%",	
     "TRB%",
     "AST%",
     "STL%",
     "BLK%",
     "TOV%",
     "ORB%",
     "FTr",
    "PTS_against", "FG_against",
      "FGA_against",
      "FG%_against",
     "3P_against",
     "3PA_against",
     "3P%_against",
     "2P_against",
     "2PA_against",
     "2P%_against",
    "eFG%_against",
     "FT_against",
    "FTA_against",
     "FT%_against",
    "ORB_against",
    "DRB_against",
     "TRB_against",
     "AST_against",
     "STL_against",
     "BLK_against",
    "TOV_against",
     "PF_against",
     "TOV%_against",
     "ORB%_against",
    "FT/FGA_against",
    "Total"
]

# =====================================================
# ROLLING MEANS (PRE-GAME)
# =====================================================
for window in rolling_windows:
    for col in stats_columns:
        df[f"Rolling_{col}_{window}"] = (
            df
            .groupby(["Team", "season"])[col]
            .rolling(window=window, min_periods=window)
            .mean()
            .shift(1)
            .reset_index(level=[0, 1], drop=True)
        )

# =====================================================
# FINAL TABLE
# =====================================================
KEEP_COLS = [
    "event_key",
    "Date",
    "season",
    "Team",
    "Opp_name",
    "Is_Home"
]

ROLLING_COLS = [c for c in df.columns if c.startswith("Rolling_")]

team_game_features = (
    df[KEEP_COLS + ROLLING_COLS]
    .dropna(subset=ROLLING_COLS)
    .round(3)
    .sort_values(["Date", "Team"])
)

team_game_features.to_csv("team_game_features.csv", index=False)
team_game_features.to_parquet(
    "team_game_features.parquet",
    compression="zstd"
)

print("Saved team_game_features")


## 4. Derived Feature Engineering

Creates additional predictive features by combining team performance, offensive and defensive metrics, pace, and recent form.

In [ ]:
#создаем derived_features.csv

import pandas as pd

df_clean = pd.read_csv("cleaned_games.csv", parse_dates=["Date"])
df_roll = pd.read_csv("team_game_features.csv", parse_dates=["Date"])

df = df_clean.merge(df_roll, on = ["event_key", "Date", "Team", "season", "Opp_name", "Is_Home"], how = "inner")


df["Derived_eFG_5"]  = df["Rolling_eFG%_5"] - df["Rolling_eFG%_against_5"]
df["Derived_TOV_5"]  = df["Rolling_TOV_5"] - df["Rolling_TOV_against_5"]
df["Derived_ORB_5"]  = df["Rolling_ORB_5"] - df["Rolling_ORB_against_5"]
df["Derived_DRB_5"]  = df["Rolling_DRB_5"] - df["Rolling_DRB_against_5"]


df["Derived_eFG_15"]  = df["Rolling_eFG%_15"] - df["Rolling_eFG%_against_15"]
df["Derived_TOV_15"]  = df["Rolling_TOV_15"] - df["Rolling_TOV_against_15"]
df["Derived_ORB_15"]  = df["Rolling_ORB_15"] - df["Rolling_ORB_against_15"]
df["Derived_DRB_15"]  = df["Rolling_DRB_15"] - df["Rolling_DRB_against_15"]

df["Derived_pace_x_ortg_5"] = df["Rolling_Pace_5"] * df["Rolling_ORtg_5"]
df["Derived_b2b_x_pace_5"]  = df["BackToBack"] * df["Rolling_Pace_5"]
df["Derived_pace_x_ortg_15"] = df["Rolling_Pace_15"] * df["Rolling_ORtg_15"]
df["Derived_b2b_x_pace_15"]  = df["BackToBack"] * df["Rolling_Pace_15"]

df["Derived_Form_Up"] = df["Rolling_PTS_for_5"] - df["Rolling_PTS_for_15"] 
df["Derived_Pace_Shift"] = df["Rolling_Pace_5"] / df["Rolling_Pace_15"]
df["Derived_Form_Up_against"] = df["Rolling_PTS_against_5"] - df["Rolling_PTS_against_15"] 


df["Derived_Off_vs_Def_5"] = (
    df["Rolling_ORtg_5"] - df["Rolling_DRtg_5"]
)

df["Derived_Off_vs_Def_15"] = (
    df["Rolling_ORtg_15"] - df["Rolling_DRtg_15"]
)

df["Derived_PTS_std_diff_5"] = (
    df["Rolling_PTS_for_std_5"] - df["Rolling_PTS_against_std_5"]
)

df["Derived_PTS_std_diff_15"] = (
    df["Rolling_PTS_for_std_15"] - df["Rolling_PTS_against_std_15"]
)

# df["Derived_Pace_std_diff_5"] = (
#     df["Rolling_Pace_std_5"] - df["Rolling_Pace_std_against_5"]
# )




ROLLING_COLS = [
    "Derived_eFG_5",
    "Derived_TOV_5",
    "Derived_ORB_5",
    "Derived_DRB_5",
    "Derived_eFG_15",
    "Derived_TOV_15",
    "Derived_ORB_15",
    "Derived_DRB_15",
    "Derived_pace_x_ortg_5",
    "Derived_b2b_x_pace_5",
    "Derived_pace_x_ortg_15",
    "Derived_b2b_x_pace_15",
    "Derived_Form_Up",
    "Derived_Pace_Shift",
    "Derived_Form_Up_against",
    "Derived_Off_vs_Def_5",
    "Derived_Off_vs_Def_15",
    "Derived_PTS_std_diff_5",
    "Derived_PTS_std_diff_15"
]

KEEP_COLS = [
    "event_key",
    "Team"
]

df = df[KEEP_COLS + ROLLING_COLS].round(2).sort_values("event_key")
df= df.dropna(subset="event_key")

df.to_csv("derived_features.csv", index=False)
df.to_parquet("derived_features.parquet", compression="zstd")
print("Saved derived_features.csv")



## 5. Historical Training Dataset

Combines engineered historical features into a dataset prepared for model training and evaluation.

In [ ]:
# ================= CREATE training_dataset =================

import pandas as pd
import re
import numpy as np
import json

# ================= HELPERS =================
def parse_spread(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    try:
        return float(x)
    except ValueError:
        pass
    if "май" in x:  # excel-bug
        num = re.findall(r"\d+", x)
        if num:
            return float(num[0]) + 0.5
    return np.nan


TEAM_RENAME_MAP = {
    "Utah": "Utah Jazz",
    "LA Lakers": "Los Angeles Lakers",
    "Houston": "Houston Rockets",
    "San Antonio": "San Antonio Spurs",
    "Portland": "Portland Trail Blazers",
    "Golden State": "Golden State Warriors",
    "New Jersey": "New Jersey Nets",
    "Dallas": "Dallas Mavericks",
    "Cleveland": "Cleveland Cavaliers",
    "Seattle": "Seattle SuperSonics",
    "Washington": "Washington Wizards",
    "Orlando": "Orlando Magic",
    "Chicago": "Chicago Bulls",
    "Sacramento": "Sacramento Kings",
    "Milwaukee": "Milwaukee Bucks",
    "Toronto": "Toronto Raptors",
    "New Orleans": "New Orleans Pelicans",
    "Memphis": "Memphis Grizzlies",
    "Denver": "Denver Nuggets",
    "Philadelphia": "Philadelphia 76ers",
    "Indiana": "Indiana Pacers",
    "Miami": "Miami Heat",
    "Detroit": "Detroit Pistons",
    "Phoenix": "Phoenix Suns",
    "New York": "New York Knicks",
    "Atlanta": "Atlanta Hawks",
    "Minnesota": "Minnesota Timberwolves",
    "Charlotte": "Charlotte Hornets",
    "Boston": "Boston Celtics"
}

# ================= LOAD =================
df_clean  = pd.read_parquet("cleaned_games.parquet")
df_roll   = pd.read_parquet("team_game_features.parquet")
df_deriv  = pd.read_parquet("derived_features.parquet")
df_market = pd.read_csv("market_aggregates.csv").rename(columns={"team": "Team"})

df_market_hist = pd.read_csv(
    "oddsData.csv",
    sep=";",
    parse_dates=["date"],
    dayfirst=True
).rename(columns={
    "date": "Date",
    "team": "Team",
    "opponent": "Opp_name",
    "spread": "Spread"
})

df_market_hist["Team"] = df_market_hist["Team"].map(TEAM_RENAME_MAP)
df_market_hist["Opp_name"] = df_market_hist["Opp_name"].map(TEAM_RENAME_MAP)

df_market_hist["total"] = pd.to_numeric(df_market_hist["total"], errors="coerce")
df_market_hist["Spread"] = df_market_hist["Spread"].apply(parse_spread)

df_market_hist["Market_TT_hist"] = (
    df_market_hist["total"] / 2
    - df_market_hist["Spread"] / 2
)

# ================= BASE =================
df = df_clean.copy()

# ================= MERGES =================
df = df.merge(
    df_roll,
    on=["event_key", "Date", "Team", "season", "Opp_name", "Is_Home"],
    how="left"
)

df = df.merge(
    df_deriv,
    on=["event_key", "Team"],
    how="left"
)

df = df.merge(
    df_market,
    on=["event_key", "Team"],
    how="left"
)

df = df.merge(
    df_market_hist[["Date", "Team", "Opp_name", "Market_TT_hist", "Spread"]],
    on=["Date", "Team", "Opp_name"],
    how="left"
)

# ================= MARKET BASE =================

# Market_TT
df["Market_TT"] = (
    df["closing_line"]
    .fillna(df["Market_TT_hist"])
)

# Total — ТОЛЬКО если есть (из истории), иначе NaN
df["Total"] = df["Total"]

# если Total нет — восстанавливаем из Market_TT и Spread_hist
df["Total"] = df["Total"].fillna(
    2 * df["Market_TT"] + df["Spread"]
)

# Spread в team-терминах (ЕДИНАЯ ФОРМУЛА)
df["Spread"] = df["Total"] - 2 * df["Market_TT"]

df["Spread_abs"] = df["Spread"].abs()
df["Is_Favorite"] = (df["Spread"] < 0).astype(int)

print(
    df[["Market_TT", "Total", "Spread"]]
    .describe()
)

# ================= TARGET =================
df["PTS_target"] = df["PTS_for"]


# ================= FILTER =================
df = df[
    df["Date"].notna() &
    df["PTS_target"].notna()
   # &    df["Market_TT"].notna()
]
df["Spread"] = df["Spread"].clip(-30, 30)
df["Total"]  = df["Total"].clip(180, 260)


# ================= COLUMNS =================
ROLLING_COLS = [c for c in df.columns if c.startswith("Rolling_")]
DERIVED_COLS = [c for c in df.columns if c.startswith("Derived_")]
MARKET_COLS = [
    "Market_TT",
    "Spread",
    "Spread_abs",
    "Is_Favorite"
]
KEEP_COLS = ["event_key", "Team", "Date", "season", "Is_Home","PTS_target", "points_diff", "HoursRest","LongRest","ShortRest","BackToBack"]

df = df[KEEP_COLS + ROLLING_COLS + DERIVED_COLS + MARKET_COLS]

#df = df.drop(columns=["Rolling_TOV_against_5","Rolling_TOV_against_15","Rolling_ORB_5","Rolling_ORB_15","Rolling_ORB_against_5","Rolling_ORB_against_15"])

# ================= DEDUP =================
df = (
    df.drop_duplicates(subset=["event_key", "Team"], keep="last")
      .sort_values("Date")
      .reset_index(drop=True)
)

# ================= CHECK =================
# 1. Нет ли NaN в rolling / derived
MIN_FEATURES = [
    "Rolling_PTS_for_5",
    "Rolling_PTS_for_15",
    "Rolling_PTS_for_std_5",
    "Rolling_PTS_for_std_15",
]

before = len(df)

df = df.dropna(subset=MIN_FEATURES)

after = len(df)

print(f"Dropped {before - after} rows due to insufficient history")

# 1. Проверка
assert df[MIN_FEATURES].isna().sum().max() == 0

# 2. Сколько матчей осталось
print("Final rows:", len(df))

# 3. Потеря по сезонам
print(
    df.groupby("season").size().describe()
)

# 2. Нет ли будущих дат
assert df["Date"].is_monotonic_increasing

# 3. Distribution check
print(df["PTS_target"].describe())

# 4. Correlation sanity
print(df[["PTS_target","Rolling_PTS_for_5","Rolling_PTS_for_std_5"]].corr())


print("TOTAL ROWS:", len(df))
print("PTS_target NaN:", df["PTS_target"].isna().sum())

n_test = int(len(df) * 0.2)
test_df = df.iloc[-n_test:]

print("\nLAST 20% CHECK")
print("Test rows:", len(test_df))
print("PTS_target NaN in test:", test_df["PTS_target"].isna().sum())

# ================= SAVE =================
df.to_parquet("training_dataset.parquet", compression="zstd")
df.to_csv("training_dataset.csv", index=False)

print("✅ Saved training_dataset")


## 6. Future Prediction Dataset

Prepares feature records for upcoming NBA games using the latest available historical and market information.

In [ ]:
# ================= CREATE future_dataset =================

import pandas as pd
import json

# ================= LOAD MARKET =================
df_market = (
    pd.read_csv("market_aggregates.csv")
      .rename(columns={"team": "Team"})
)

# Date из event_key
df_market["Date"] = pd.to_datetime(
    df_market["event_key"].str[:10],
    errors="coerce"
)

last_date = df_market["Date"].max()

df_future_base = df_market[df_market["Date"] == last_date].copy()

print("Future games:", len(df_future_base))
print("Date:", last_date)

# ================= LOAD TRAIN =================
df_train = pd.read_parquet("training_dataset.parquet")
df_train["Date"] = pd.to_datetime(df_train["Date"])

df_train_hist = df_train[df_train["Date"] < last_date].sort_values("Date")

# ================= FORM FEATURES =================
ROLLING_COLS = [c for c in df_train_hist.columns if c.startswith("Rolling_")]
DERIVED_COLS = [c for c in df_train_hist.columns if c.startswith("Derived_")]

FORM_COLS = ["Team"] + ROLLING_COLS + DERIVED_COLS

last_team_form = (
    df_train_hist[FORM_COLS + ["Date"]]
    .sort_values("Date")
    .groupby("Team", as_index=False)
    .tail(1)
    .drop(columns=["Date"])
)

# ================= MERGE FORM =================
df_future = df_future_base.merge(
    last_team_form,
    on="Team",
    how="left"
)

# # ================= LOAD STATS =================
# with open("training_market_stats.json", "r") as f:
#     stats = json.load(f)

# # ================= MARKET FEATURES =================
# df_future["Market_TT"] = df_future["closing_line"]
# df_future["Total"] = df_future["line_median"]

# # ⬅️ ВАЖНО: всегда создаем Spread заново
# df_future["Spread"] = df_future["Total"] - df_future["Market_TT"]

# df_future["Spread_abs"] = df_future["Spread"].abs()
# df_future["Is_Favorite"] = (df_future["Spread"] < 0).astype(int)

# df_future["Market_TT_z"] = (
#     df_future["Market_TT"] - stats["Market_TT_mean"]
# ) / stats["Market_TT_std"]

# df_future["Total_z"] = (
#     df_future["Total"] - stats["Total_mean"]
# ) / stats["Total_std"]
# ================= CLEAN =================
DROP_COLS = [
    c for c in df_future.columns
    if c.endswith("_hist")
    or c.startswith("event_key_")
    or c.startswith("Date_")
    or c in ["opening_line","closing_line","line_median","line_std","odds_median","book_count"]
]

df_future = df_future.drop(columns=DROP_COLS, errors="ignore")

assert "PTS_target" not in df_future.columns

print("TOTAL FUTURE ROWS:", len(df_future))

# ================= SAVE =================
df_future.to_parquet("future_dataset.parquet", compression="zstd")
df_future.to_csv("future_dataset.csv", index=False)

print("✅ Saved future_dataset")
